# Decision Tree From Scratch vs Scikit-learn
A Decision Tree is a greedy, recursive partitioning algorithm that splits data to minimize error.

- For classification → minimize impurity (Gini / Entropy)
- For regression → minimize variance (MSE)

In [52]:

import pandas as pd
from pandas_datareader import data
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV

In [53]:
df= pd.read_csv('HousingData.csv')

In [54]:
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,NaN,36.2


In [55]:
X = df.iloc[:,0:13].values
y = df.iloc[:,-1].values

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=4)

In [57]:
rt = DecisionTreeRegressor(criterion = 'squared_error', max_depth=5)

In [58]:
rt.fit(X_train,y_train)

DecisionTreeRegressor(max_depth=5)

In [59]:
y_pred = rt.predict(X_test)

In [60]:
r2_score(y_test,y_pred)

0.6511211906427593

In [61]:
from sklearn.model_selection import cross_val_score, KFold

In [62]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [63]:
cv_scores = cross_val_score(rt, X, y, cv=kf, scoring='r2')

print("CV R2 Scores for each fold:", cv_scores)
print("Mean CV R2 Score:          ", cv_scores.mean())
print("Standard Deviation:        ", cv_scores.std())

CV R2 Scores for each fold: [0.72417698 0.75701667 0.65131349 0.69780836 0.79971166]
Mean CV R2 Score:           0.7260054333672455
Standard Deviation:         0.05054029712241513


In [64]:
from sklearn.model_selection import GridSearchCV, KFold

# Parameter grid to search
param_grid = {
    'max_depth'        : [1,2, 3, 4, 5, 6, 7, 8, 9 , 10],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 4, 8]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    DecisionTreeRegressor(),
    param_grid,
    cv=kf,
    scoring='r2',
    verbose=1
)

grid_search.fit(X, y)

print("Best Parameters:", grid_search.best_params_)
print("Best CV R2:     ", grid_search.best_score_)

Fitting 5 folds for each of 160 candidates, totalling 800 fits
Best Parameters: {'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 20}
Best CV R2:      0.7591072657715194


In [65]:
# Train best model on full training data
best_model = grid_search.best_estimator_

# Predict on test set
y_pred_tuned = best_model.predict(X_test)

print("Tuned Test R2:    ", r2_score(y_test, y_pred_tuned))
print("Tuned CV R2:      ", grid_search.best_score_)

Tuned Test R2:     0.9271968515254447
Tuned CV R2:       0.7591072657715194


## Hyperparameter Tuning

In [66]:
param_grid = {
    'max_depth':[2,4,8,10,None],
    'criterion':['msquared_error','absolute_error'],
    'max_features':[0.25,0.5,1.0],
    'min_samples_split':[0.25,0.5,1.0]
}
     

In [67]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('reg', DecisionTreeRegressor())
])

param_grid = {
    'reg__max_depth': [2, 4, 8, 10, None],
    'reg__criterion': ['squared_error', 'absolute_error'],
    'reg__max_features': [0.25, 0.5, 1.0],
    'reg__min_samples_split': [0.25, 0.5, 1.0]
}


reg = GridSearchCV(pipe, param_grid, cv=5)
reg.fit(X_train, y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('imputer', SimpleImputer()),
                                       ('reg', DecisionTreeRegressor())]),
             param_grid={'reg__criterion': ['squared_error', 'absolute_error'],
                         'reg__max_depth': [2, 4, 8, 10, None],
                         'reg__max_features': [0.25, 0.5, 1.0],
                         'reg__min_samples_split': [0.25, 0.5, 1.0]})

In [68]:
reg.best_score_

0.7006358806155208

In [69]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None


# GINI
it is used for a classfier tree
- we just have to replace our loss function

In [70]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs ** 2)


## Regression Tree Criterion

Mean Squared Error (MSE):

MSE = (1/n) * Σ (yᵢ - ȳ)²

- Lower MSE → better split  
- Leaf node prediction = mean of target values  

In [71]:
def mse(y):
    mean = np.mean(y)
    return np.mean((y - mean) ** 2)

## Custom Implementation Highlights

- Uses brute-force split search  
- Thresholds = unique feature values  
- Leaf value = mean(y)  
- Fully recursive tree construction  


In [72]:
def best_split(X, y):
    best_mse, best_feat, best_thresh = float('inf'), None, None
    n_samples, n_features = X.shape

    for feat in range(n_features):
        thresholds = np.unique(X[:, feat])
        for thresh in thresholds:
            left_mask = X[:, feat] <= thresh
            right_mask = ~left_mask

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            mse_split = (
                left_mask.sum() * mse(y[left_mask]) +
                right_mask.sum() * mse(y[right_mask])
            ) / n_samples

            if mse_split < best_mse:
                best_mse = mse_split
                best_feat = feat
                best_thresh = thresh

    return best_feat, best_thresh

In [73]:
def build_tree(X, y, depth, max_depth, min_samples_split):
    if len(y) < min_samples_split or depth == max_depth:
        return Node(value=np.mean(y))

    feat, thresh = best_split(X, y)

    if feat is None:
        return Node(value=np.mean(y))

    left_mask = X[:, feat] <= thresh
    right_mask = ~left_mask

    left = build_tree(X[left_mask], y[left_mask], depth + 1, max_depth, min_samples_split)
    right = build_tree(X[right_mask], y[right_mask], depth + 1, max_depth, min_samples_split)

    return Node(feature=feat, threshold=thresh, left=left, right=right)


In [74]:
class MyDecisionTreeRegressor:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = build_tree(X, y, 0, self.max_depth, self.min_samples_split)

    def _traverse(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

## Comparison Setup

- Same dataset (Boston Housing)
- Same train-test split  
- Same hyperparameters:
  - max_depth = 10  
  - min_samples_split = 2  


In [75]:
custom_dt = MyDecisionTreeRegressor(max_depth=10, min_samples_split=2)
custom_dt.fit(X_train, y_train)

In [76]:
y_pred = custom_dt.predict(X_test)

In [78]:
sk_reg = DecisionTreeRegressor(max_depth=10, min_samples_split=2)
sk_reg.fit(X_train, y_train)

sk_pred = sk_reg.predict(X_test)

In [80]:
from sklearn.metrics import r2_score, mean_squared_error
print("Custom R2:", r2_score(y_test, y_pred))
print("Sklearn R2:", r2_score(y_test, sk_pred))

print("Custom MSE:", mean_squared_error(y_test, y_pred))
print("Sklearn MSE:", mean_squared_error(y_test, sk_pred))

Custom R2: 0.7059906092579948
Sklearn R2: 0.6039014230896351
Custom MSE: 27.310304253758115
Sklearn MSE: 36.793289570111554


## Observations

- Custom model performed better on this split  
- Predictions are similar but not identical  
- Differences arise due to:
  - Different split thresholds  
  - Greedy nature of trees  
  - High variance  


## Key Insights

- Decision Trees are high variance models  
- Small implementation differences → large output differences  
- Better performance on one split does not guarantee generalization  


In [81]:
diff = np.abs(y_pred - sk_pred)

print("Mean Absolute Difference:", np.mean(diff))
print("Max Difference:", np.max(diff))

Mean Absolute Difference: 2.3903647649382274
Max Difference: 23.1


## Limitations of Custom Model

- No pruning  
- Inefficient split search (O(n²))  
- No handling of missing values  
- No regularization  
